In [ ]:
# run.ipynb
# Pull latest code before running, then run build/embed/train.
REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
LOG = "/workspace/stable_query_latent_logs/pipeline.log"
print('repo:', REPO)
print('log :', LOG)


In [ ]:
# Clone or pull before every run.
import os

if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}

%cd {REPO}
!git remote set-url origin {URL}
!git remote -v
!git pull origin main
!git status --short
!git rev-parse HEAD


In [ ]:
# Start GPU + CPU + RAM + disk I/O monitor.
import subprocess, threading, time, psutil
from pathlib import Path

stop = False

def _read_int(path):
    try:
        text = Path(path).read_text().strip()
        if text == "max":
            return None
        return int(text)
    except Exception:
        return None

def get_memory_status():
    limit = _read_int("/sys/fs/cgroup/memory.max")
    used = _read_int("/sys/fs/cgroup/memory.current")
    if limit is None or used is None:
        limit = _read_int("/sys/fs/cgroup/memory/memory.limit_in_bytes")
        used = _read_int("/sys/fs/cgroup/memory/memory.usage_in_bytes")
    if limit and used and limit < 10**18:
        return used / limit * 100, used / 1024**3, limit / 1024**3, "cgroup"
    vm = psutil.virtual_memory()
    return vm.percent, vm.used / 1024**3, vm.total / 1024**3, "host"

def get_gpu_status():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            timeout=3,
        ).stdout.strip()
        parts = [p.strip() for p in out.splitlines()[0].split(",")]
        return f"{float(parts[0]):.0f}%, {float(parts[1])/1024:.1f}/{float(parts[2])/1024:.1f} GiB"
    except Exception as e:
        return f"n/a ({e})"

def monitor(interval=5):
    last_disk = psutil.disk_io_counters()
    last_t = time.time()
    psutil.cpu_percent(interval=None)
    while not stop:
        gpu = get_gpu_status()
        cpu = psutil.cpu_percent(interval=None)
        ram_pct, ram_used, ram_total, ram_source = get_memory_status()
        now_disk = psutil.disk_io_counters()
        now_t = time.time()
        dt = max(now_t - last_t, 1e-6)
        read_mb = (now_disk.read_bytes - last_disk.read_bytes) / 1e6 / dt
        write_mb = (now_disk.write_bytes - last_disk.write_bytes) / 1e6 / dt
        last_disk, last_t = now_disk, now_t
        print(
            f"[gpu] {gpu} | [cpu] {cpu:.0f}% | "
            f"[ram:{ram_source}] {ram_pct:.0f}% ({ram_used:.1f}/{ram_total:.1f} GiB) | "
            f"[disk] R {read_mb:.1f} MB/s W {write_mb:.1f} MB/s",
            flush=True,
        )
        time.sleep(interval)

threading.Thread(target=monitor, daemon=True).start()


In [ ]:
# Build text H5.
!python -u game_review_data/build.py   --data-dir game_review_data/build_new_gamedata   --only metadata split text-h5   --split-device cuda   --logout-address {LOG}


In [ ]:
# Clear GPU memory.
import gc, torch
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Embed text H5.
!python -u game_review_data/embedding_incloud.py   --input-h5 game_review_data/build_new_gamedata/text_h5.h5   --output-h5 game_review_data/embedding_h5.h5   --device cuda   --text-load-chunk-size 500000   --tok-workers 4   --logout-address {LOG}


In [ ]:
# Clear GPU memory.
import gc, torch
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Train full sweep (TRAINING ONLY).
# Probes are deferred to eval.ipynb to avoid GPU contention during training:
#   * --probe-queue-dir + --train-probe-every keep emitting slim probe
#     checkpoints every few epochs (cheap, no probe GPU work here).
#   * NO probe_worker runs in this notebook; the queue just accumulates on disk.
#   * --eval-mode none skips the in-sweep final eval (also moved to eval.ipynb).
# OOM-budget calibration runs automatically at startup (--calib-mode measure,
# default): it fits peak=R+C*S per num_latents via pseudo batches and plans
# backward-mode / paired / sentence caps per combo. Tip: run a dry-run plan
# first with `python VICReg_review/oom_proxy.py --h5 game_review_data/embedding_h5.h5
#   --calib-out VICReg_review/heads/cloud_full_sweep_a100/calib.json --measure --plan`.
!python -u VICReg_review/sweep_cloud.py \
  --h5 game_review_data/embedding_h5.h5 \
  --out-dir VICReg_review/heads/cloud_full_sweep_a100 \
  --train-game-counts 50 100 200 500 1000 1500 2000 \
  --sample-fractions 0.8 0.6 0.4 0.2 \
  --output-dims 18 36 64 72 \
  --latent-scales 1 2 4 \
  --base-num-latents 256 \
  --expander-dim 128 \
  --expander-hidden 128 \
  --epochs 30 \
  --batch-size 128 \
  --cache-mode full \
  --prefetch-batches 2 \
  --cache-dtype float16 \
  --pin-cache \
  --train-probe-every 5 \
  --probe-queue-dir VICReg_review/heads/cloud_full_sweep_a100/probe_queue \
  --eval-mode none \
  --train-game-anchor-appids "1091500,1385380" \
  --logout-address {LOG}


In [ ]:
# Training done. Stop the monitor and print the sweep status.
# Probe draining + final eval + artifact collection now live in eval.ipynb
# (run it next, on this same pod, after training finishes).
stop = True

import json
from pathlib import Path

sweep_manifest = Path(REPO, 'VICReg_review/heads/cloud_full_sweep_a100/sweep_manifest.json')
queue_dir = Path(REPO, 'VICReg_review/heads/cloud_full_sweep_a100/probe_queue')

if sweep_manifest.exists():
    data = json.loads(sweep_manifest.read_text(encoding='utf-8'))
    print('sweep status:', data.get('status'), '| updated:', data.get('updated_at'))
    rows = data.get('rows')
    if isinstance(rows, list):
        done = sum(1 for r in rows if r.get('status') == 'done')
        print(f'combinations done: {done}/{len(rows)}')
    if data.get('error'):
        print('error:', data['error'])
else:
    print('no sweep_manifest yet:', sweep_manifest)

pending = sorted(queue_dir.glob('*.json')) if queue_dir.exists() else []
print(f'probe jobs queued (drain these in eval.ipynb): {len(pending)}')
print('Next: run Pod/eval.ipynb on this pod to drain probes, run final eval, and archive.')
